# Support Vector Machines with SMS++

**SMS++** models the training problem of a Support Vector Machine as a `SVCBlock`
(classification) or a `SVRBlock` (regression), and solves it with any of its solvers:
the ad hoc `SMOSolver`, a general-purpose MILP solver such as Gurobi, or a
`LagrangianDualSolver` working on the problem split in chunks.

This example follows the workflow one would follow with **scikit-learn**, and runs the
two side by side: the same train/test split, the same k-fold cross-validation, the same
grid of hyper-parameters, and the same scores. The point is that the model is the same
one, so that SMS++ can be used wherever scikit-learn would be, with the added freedom
of choosing how the training problem is written and which solver attacks it.

The notebook needs the `svm_solver` tool of SMS++ on the `PATH`.

In [ ]:
import os
import tempfile

import numpy as np
import pandas as pd

from pysmspp import Block, SMSConfig, SMSFileType, SMSNetwork, SVMSolver, Variable

assert SVMSolver().is_available(), "svm_solver is not in the PATH"

## The data set

The Wisconsin breast cancer data set, one of those bundled with scikit-learn, is a
binary classification problem over 30 features. SMS++ wants the labels to be `-1` and
`+1`, which is the convention of the maximum-margin formulation.

A stratified subsample keeps the notebook quick: `SMOSolver` is not yet tuned for large
data sets, and a few hundred samples are enough to make the point. The features are
standardised, as they always should be for a SVM, and a stratified split holds out a
third of the samples for the final evaluation: the test set is touched **once**, at the
end, and the hyper-parameters are chosen by cross-validation on the training set alone.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
y = np.where(y == 1, 1.0, -1.0)  # SMS++ wants the labels in {-1,+1}

X, _, y, _ = train_test_split(X, y, train_size=300, stratify=y, random_state=0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1 / 3, stratify=y, random_state=0
)

scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

print(
    f"{X_train.shape[0]} training and {X_test.shape[0]} test samples"
    f" of {X_train.shape[1]} features"
)

## A scikit-learn estimator backed by SMS++

Everything scikit-learn does around a model, cross-validation, grid search, pipelines,
learning curves, works on anything that looks like an estimator. The class below is
that: `fit()` writes the data set as a `SVCBlock` (or a `SVRBlock`) into a netCDF file,
runs `svm_solver` on it and reads the trained model back out of the `SVMBlockSolution`
that the tool writes, while `decision_function()` evaluates the kernel expansion
`f(x) = sum_i c_i K(x_i, x) + b`, which is the form the model always has, whatever
formulation was solved. The coefficients are `c_i = y_i alpha_i` for a classifier and
`c_i = alpha_i - alpha_(n+i)` for a regressor, the two sides of the tube.

The class is deliberately short and lives in the notebook: it is the whole of what it
takes to plug SMS++ into scikit-learn.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin

KERNELS = {"linear": 0, "poly": 1, "gaussian": 2, "laplacian": 3, "sigmoid": 4}
GAMMAS = {"scale": 0.0, "auto": -1.0}  # the two values SMS++ derives from the data
INTEGERS = ("Kernel", "Degree", "SquaredLoss", "RegBias")


class SMSppSVM(BaseEstimator):
    """A scikit-learn estimator training a SVMBlock with SMS++."""

    _block_type = "SVMBlock"

    def __init__(
        self,
        C=1.0,
        kernel="linear",
        gamma="scale",
        degree=3,
        coef0=0.0,
        epsilon=0.1,
        squared_loss=False,
        reg_bias=False,
        solver="SVMBlock/SVMSCfg.txt",
        block_config=None,
        chunks=1,
    ):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.epsilon = epsilon
        self.squared_loss = squared_loss
        self.reg_bias = reg_bias
        self.solver = solver
        self.block_config = block_config
        self.chunks = chunks

    # the hyper-parameters, in the netCDF names of the SVMBlock

    def _hyperparameters(self):
        hp = {
            "C": float(self.C),
            "Kernel": KERNELS[self.kernel],
            "Gamma": GAMMAS.get(self.gamma, self.gamma),
            "Degree": int(self.degree),
            "Coef0": float(self.coef0),
            "SquaredLoss": int(self.squared_loss),
            "RegBias": int(self.reg_bias),
        }
        if self._block_type == "SVRBlock":
            hp["Epsilon"] = float(self.epsilon)
        return hp

    def _gamma(self, X):  # the value SMS++ uses, needed to evaluate the kernel here
        if self.gamma == "scale":
            return 1.0 / (X.shape[1] * X.var())
        if self.gamma == "auto":
            return 1.0 / X.shape[1]
        return float(self.gamma)

    def _kernel(self, A, B):
        code, gamma = KERNELS[self.kernel], self.gamma_
        if code == 0:
            return A @ B.T
        if code == 1:
            return (gamma * (A @ B.T) + self.coef0) ** self.degree
        if code == 2:
            return np.exp(-gamma * ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1))
        if code == 3:
            return np.exp(-gamma * np.abs(A[:, None, :] - B[None, :, :]).sum(-1))
        return np.tanh(gamma * (A @ B.T) + self.coef0)

    def _network(self, X, y):  # the data set and the hyper-parameters, as a Block
        block = Block().from_kwargs(
            block_type=self._block_type,
            NSamples=X.shape[0],
            NFeatures=X.shape[1],
            X=Variable("X", "double", ("NSamples", "NFeatures"), np.asarray(X, float)),
            Y=Variable("Y", "double", ("NSamples",), np.asarray(y, float)),
            **{
                name: Variable(name, "int" if name in INTEGERS else "double", (), value)
                for name, value in self._hyperparameters().items()
            },
        )
        network = SMSNetwork(file_type=SMSFileType.eBlockFile)
        network.add(self._block_type, "Block_0", block=block)
        return network

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)

        network = self._network(X, y)

        folder = tempfile.mkdtemp()
        fp_network = os.path.join(folder, "svm.nc4")
        fp_solution = os.path.join(folder, "model.nc4")
        network.to_netcdf(fp_network, force=True)

        options = {} if self.chunks == 1 else {"s": self.chunks}
        if self.block_config is not None:
            options["B"] = self.block_config

        self.solver_ = SVMSolver(
            fp_network=fp_network,
            configfile=str(SMSConfig(template=self.solver)),
            fp_solution=fp_solution,
            **options,
        )
        self.solver_.optimize(logging=False)

        model = self.solver_.solution.blocks["Solution_0"]
        alphas = np.asarray(model.variables["Multipliers"].data)

        self.X_, self.b_ = X, float(model.variables["Bias"].data)
        self.gamma_ = self._gamma(X)
        self.classes_ = np.unique(y)
        self.objective_value_ = self.solver_.objective_value

        # the coefficients of the kernel expansion, out of the multipliers
        n = X.shape[0]
        self.coef_ = y * alphas if alphas.size == n else alphas[:n] - alphas[n:]
        self.support_ = np.flatnonzero(np.abs(self.coef_) > 1e-8)

        return self

    def decision_function(self, X):
        return self._kernel(np.asarray(X, dtype=float), self.X_) @ self.coef_ + self.b_


class SMSppSVC(SMSppSVM, ClassifierMixin):
    """Support Vector Classification with SMS++."""

    _block_type = "SVCBlock"

    def predict(self, X):
        return np.sign(self.decision_function(X))


class SMSppSVR(SMSppSVM, RegressorMixin):
    """Support Vector Regression with SMS++."""

    _block_type = "SVRBlock"

    def predict(self, X):
        return self.decision_function(X)

## The same model as scikit-learn

The classifier of scikit-learn solves the very same problem, so the two must agree, not
only on the score but on the model itself: on the decision function, on which samples
end up supporting it and on the bias. They do, to the tolerance the solvers are stopped
at.

In [ ]:
from sklearn.svm import SVC

rows = []
for kernel, C in [("linear", 1.0), ("linear", 10.0), ("gaussian", 1.0)]:
    smspp = SMSppSVC(C=C, kernel=kernel).fit(X_train, y_train)
    sklearn = SVC(
        C=C, kernel="linear" if kernel == "linear" else "rbf", gamma="scale"
    ).fit(X_train, y_train)

    f_smspp = smspp.decision_function(X_test)
    f_sklearn = sklearn.decision_function(X_test)

    rows.append(
        {
            "kernel": kernel,
            "C": C,
            "test accuracy (SMS++)": smspp.score(X_test, y_test),
            "test accuracy (scikit-learn)": sklearn.score(X_test, y_test),
            "support vectors (SMS++)": smspp.support_.size,
            "support vectors (scikit-learn)": sklearn.n_support_.sum(),
            "max |f - f|": np.abs(f_smspp - f_sklearn).max(),
            "bias difference": abs(smspp.b_ - sklearn.intercept_[0]),
        }
    )

pd.DataFrame(rows)

## The same library, from both sides

`scikit-learn` trains its SVM with [LIBSVM](https://www.csie.ntu.edu.tw/~cjlin/libsvm/),
and so does SMS++ when the `SVCBlock` is given to `LIBSVMSolver`: the very same library,
called once from Python and once from C++. `libsvm-official`, the binding the authors of
LIBSVM ship (`pip install libsvm-official`), is the third way of calling it, and the
three have to give the same model, as does the ad hoc `SMOSolver`, which implements the
same algorithm without being the same code.

That is what makes the comparison an apples-to-apples one: what changes between the rows
below is only who drives the training, not what is being trained, and what is reported
is how far each of them lands from the decision function of `SMOSolver`. The residual
difference is the stopping tolerance and nothing else: the two SMS++ configurations stop
at `1e-6`, while `scikit-learn` and `libsvm-official` are left at the `1e-3` LIBSVM
defaults to.


In [ ]:
try:
    from libsvm.svmutil import svm_parameter, svm_predict, svm_problem, svm_train

    libsvm_available = True
except ImportError:  # pip install libsvm-official
    libsvm_available = False

# whether this SMS++ has been built with LIBSVM: a Solver that is not in the factory
# makes the run fail, so it is probed once here rather than at every row
probe = SMSppSVC(solver="SVMBlock/SVMSCfg-libsvm.txt")
libsvm_solver_available = True
try:
    probe.fit(X_train[:20], y_train[:20])
except RuntimeError:
    libsvm_solver_available = False

rows = []
for kernel, C in [("linear", 1.0), ("gaussian", 1.0)]:
    smo = SMSppSVC(C=C, kernel=kernel).fit(X_train, y_train)
    reference = smo.decision_function(X_test)

    row = {"kernel": kernel, "C": C}

    # LIBSVMSolver is in the Solver factory only when SMS++ has been built with LIBSVM
    if libsvm_solver_available:
        lsvm = SMSppSVC(C=C, kernel=kernel, solver="SVMBlock/SVMSCfg-libsvm.txt").fit(
            X_train, y_train
        )
        row["SMS++ / LIBSVMSolver"] = np.abs(
            lsvm.decision_function(X_test) - reference
        ).max()

    sklearn = SVC(
        C=C, kernel="linear" if kernel == "linear" else "rbf", gamma="scale"
    ).fit(X_train, y_train)
    row["scikit-learn"] = np.abs(sklearn.decision_function(X_test) - reference).max()

    if libsvm_available:
        # -t 0 is the linear kernel and -t 2 the gaussian one, and the gamma is the
        # "scale" of scikit-learn and of SMS++. No sign correction is needed: with
        # labels -1 and +1 LIBSVM makes sure that +1 is the positive class, whatever
        # the order the two are met in; SMS++ checks that rather than assuming it.
        gamma = 1.0 / (X_train.shape[1] * X_train.var())
        model = svm_train(
            svm_problem(y_train.tolist(), X_train.tolist()),
            svm_parameter(
                f"-s 0 -t {0 if kernel == 'linear' else 2} -c {C} -g {gamma:.17g} -q"
            ),
        )
        _, _, values = svm_predict(
            y_test.tolist(), X_test.tolist(), model, options="-q"
        )
        row["libsvm-official"] = np.abs(
            np.array([v[0] for v in values]) - reference
        ).max()

    rows.append(row)

# how far each of them is from the decision function of SMS++ / SMOSolver
pd.DataFrame(rows)

## The same folds on both sides

Comparing two cross-validations only means something if they run on the same folds.
The splits of `svm_solver` are drawn out of the seed `e`, and their rule is specified
down to the bit precisely so that it can be reproduced elsewhere: a *splitmix64*
sequence from the seed, an unbiased draw by rejection, a downward Fisher-Yates shuffle,
and then each class dealt out to the folds round-robin, which is what keeps them
stratified. Reproducing it here is a dozen lines, and it is what makes the numbers
below comparable rather than merely similar.

In [ ]:
MASK = (1 << 64) - 1


def shuffled_indices(n, seed):
    """The permutation of ml_utils: splitmix64 and a downward Fisher-Yates."""
    state = seed

    def draw():  # splitmix64
        nonlocal state
        state = (state + 0x9E3779B97F4A7C15) & MASK
        z = state
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & MASK
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & MASK
        return z ^ (z >> 31)

    def below(bound):  # unbiased, by rejection
        threshold = ((1 << 64) - bound) % bound
        while True:
            r = draw()
            if r >= threshold:
                return r % bound

    index = list(range(n))
    for i in range(n - 1, 0, -1):
        j = below(i + 1)
        index[i], index[j] = index[j], index[i]
    return index


def smspp_folds(y, k, seed=1):
    """The stratified k folds of ml_utils, as scikit-learn wants them."""
    order = shuffled_indices(len(y), seed)

    by_class = {}
    for i in order:
        by_class.setdefault(y[i], []).append(i)

    folds, start = [[] for _ in range(k)], 0
    for label in sorted(by_class):  # the classes in the order of their label
        group = by_class[label]
        for t, i in enumerate(group):
            folds[(start + t) % k].append(i)
        start = (start + len(group)) % k

    return [(np.setdiff1d(np.arange(len(y)), fold), np.array(fold)) for fold in folds]


folds = smspp_folds(y_train, k=5)

print(" ".join(f"fold {f}: {len(test)} samples" for f, (_, test) in enumerate(folds)))

## Model selection

With an estimator in hand, the model selection is the scikit-learn one, unchanged: a
grid of hyper-parameters compared by 5-fold cross-validation on the training set, the
best one refitted on the whole of it. The folds are the ones above, so that SMS++ is
asked exactly what `svm_solver` will be asked later. Every one of the 40 fits below is
a SMS++ solve.

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    SMSppSVC(),
    param_grid={"C": [0.1, 1.0, 10.0, 100.0], "kernel": ["linear", "gaussian"]},
    cv=folds,
    refit=True,
    return_train_score=True,
).fit(X_train, y_train)

results = pd.DataFrame(grid.cv_results_)[
    [
        "param_C",
        "param_kernel",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].rename(
    columns={
        "param_C": "C",
        "param_kernel": "kernel",
        "mean_test_score": "mean_val_score",
        "std_test_score": "std_val_score",
        "rank_test_score": "rank_val_score",
    }
)

results.sort_values(by="rank_val_score")

In [ ]:
best = grid.best_estimator_

print(f"best hyper-parameters: {grid.best_params_}")
print(f"cross-validation accuracy: {grid.best_score_:.4f}")
print(f"test accuracy:             {best.score(X_test, y_test):.4f}")
print(f"support vectors:           {best.support_.size} of {X_train.shape[0]} samples")

## The model selection of SMS++ itself

The grid search above is scikit-learn's: it drives the estimator, which runs SMS++ once
per fit. `svm_solver` also does the whole thing on its own, in one call, and there it is
a cartesian product of independent problems, so the models are trained in parallel:
`k` is the number of folds, `g` the grid and `j` how many models to train at a time.

It is the same model selection on the same grid and, since the folds above are the ones
it draws out of the same seed, on the same folds: the scores are therefore the same
numbers, not merely similar ones.

In [ ]:
fp_network = os.path.join(tempfile.mkdtemp(), "svm.nc4")
SMSppSVC()._network(X_train, y_train).to_netcdf(fp_network, force=True)

selection = SVMSolver(
    fp_network=fp_network,
    configfile=str(SMSConfig(template="SVMBlock/SVMSCfg.txt")),
    k=5,
    g="C=0.1,1,10,100",
    e=1,  # the seed of the folds, the one smspp_folds() was called with
    j=os.cpu_count(),
)
selection.optimize(logging=False)

# scikit-learn compared the gaussian kernel too, the SVCBlock above is linear
sklearn_side = results[results.kernel == "linear"][["C", "mean_val_score"]]

comparison = pd.DataFrame(
    [
        {"C": point["params"]["C"], "mean_val_score": point["score"]}
        for point in selection.scores
    ]
).merge(
    sklearn_side.astype({"C": float}),
    on="C",
    suffixes=(" (svm_solver)", " (scikit-learn)"),
)

comparison

In [ ]:
for column in comparison.columns[1:]:
    best = comparison.loc[comparison[column].idxmax()]
    print(
        f"best C by {column[len('mean_val_score ') :]:16s} {best.C:6g}"
        f"  ({column.split()[0]} = {best[column]:.4f})"
    )

## The same problem, four ways

## The same problem, five ways

What SMS++ adds to the picture is that *how* the training problem is written and *which*
solver attacks it are choices of whoever solves, not properties of the model. The
`SVCBlock` above can be given to:

- the ad hoc `SMOSolver`, which works on the Wolfe dual without even generating the
  abstract representation;
- a MILP solver such as Gurobi, on the Wolfe dual;
- the same MILP solver on the primal, which exists for the linear kernel;
- a `LagrangianDualSolver`, on the problem given the *consensus structure*, i.e., the
  samples dealt out to chunks tied by consensus constraints, which is its Dantzig-Wolfe
  decomposition;
- `LIBSVMSolver`, which hands the problem over to LIBSVM, the reference implementation
  of the very algorithm `SMOSolver` implements, and is in the Solver factory only when
  SMS++ has been built with it.

All five solve the same problem, hence they agree on its value: the Wolfe dual is
written as the maximisation that strong duality makes equal to the primal.

Two chunks, and a small instance, keep the last run quick: each chunk carries its own
share of the regularisation term, so the finer the split the flatter its subproblem and
the harder the Lagrangian dual is to close.

In [ ]:
folder = tempfile.mkdtemp()
fp_network = os.path.join(folder, "svm.nc4")

subset = slice(0, 60)  # a small instance: the four runs must give the same number
network = SMSNetwork(file_type=SMSFileType.eBlockFile)
network.add(
    "SVCBlock",
    "Block_0",
    block=Block().from_kwargs(
        block_type="SVCBlock",
        NSamples=X_train[subset].shape[0],
        NFeatures=X_train.shape[1],
        X=Variable("X", "double", ("NSamples", "NFeatures"), X_train[subset]),
        Y=Variable("Y", "double", ("NSamples",), y_train[subset]),
        C=Variable("C", "double", (), 1.0),
    ),
)
network.to_netcdf(fp_network, force=True)

runs = {
    "Wolfe dual, SMOSolver": ("SVMBlock/SVMSCfg.txt", {}),
    "Wolfe dual, Gurobi": ("SVMBlock/SVMSCfg_grb.txt", {}),
    "primal, Gurobi": ("SVMBlock/SVMSCfg_grb.txt", {"B": "SVMCfg-primal.txt"}),
    "2 chunks, LagrangianDualSolver": ("SVMBlock/SVMSCfg-LD.txt", {"s": 2}),
    "Wolfe dual, LIBSVMSolver": ("SVMBlock/SVMSCfg-libsvm.txt", {}),
}

rows = []
for name, (template, options) in runs.items():
    solver = SVMSolver(
        fp_network=fp_network, configfile=str(SMSConfig(template=template)), **options
    )
    solver.optimize(logging=False)

    # LIBSVMSolver is only there when SMS++ has been built with LIBSVM
    if "LIBSVM" in name and "Success" not in solver.status:
        continue

    rows.append(
        {
            "formulation and solver": name,
            "status": solver.status,
            "training problem": solver.objective_value,
        }
    )

pd.DataFrame(rows)

## Regression

The regression variant is the same story: a `SVRBlock`, the epsilon-insensitive loss and
the same workflow, on the diabetes data set. The score is the coefficient of
determination `R2`, the larger the better, which is what both `RegressorMixin` and
`svm_solver` report.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.svm import SVR

Xr, yr = load_diabetes(return_X_y=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=1 / 3, random_state=0
)

scaler = StandardScaler().fit(Xr_train)
Xr_train, Xr_test = scaler.transform(Xr_train), scaler.transform(Xr_test)

smspp = SMSppSVR(C=100.0, epsilon=5.0, kernel="linear").fit(Xr_train, yr_train)
sklearn = SVR(C=100.0, epsilon=5.0, kernel="linear").fit(Xr_train, yr_train)

pd.DataFrame(
    [
        {
            "estimator": "SMS++",
            "test R2": smspp.score(Xr_test, yr_test),
            "support vectors": smspp.support_.size,
        },
        {
            "estimator": "scikit-learn",
            "test R2": sklearn.score(Xr_test, yr_test),
            "support vectors": sklearn.support_.size,
        },
    ]
)

In [ ]:
grid = GridSearchCV(
    SMSppSVR(kernel="linear"),
    param_grid={"C": [1.0, 10.0, 100.0], "epsilon": [1.0, 5.0]},
    cv=5,
).fit(Xr_train, yr_train)

print(f"best hyper-parameters: {grid.best_params_}")
print(f"cross-validation R2:   {grid.best_score_:.4f}")
print(f"test R2:               {grid.best_estimator_.score(Xr_test, yr_test):.4f}")

## Conclusions

The two libraries agree on the model, so a SVM trained with SMS++ can be used exactly
where a scikit-learn one would be, and the scikit-learn machinery around it, splits,
cross-validation, grid search, keeps working as it is.

What SMS++ adds is on the other side: the training problem is a `Block`, and how it is
written and solved is a configuration matter. The Wolfe dual, the primal and the
decomposition into chunks tied by consensus constraints are the same problem, and the
ad hoc `SMOSolver`, a general-purpose MILP solver and a `LagrangianDualSolver` are three
ways of solving it, which is what makes the same model reachable from very different
algorithmic directions.